In [2]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# Iniciar uma sessão Spark
spark = SparkSession.builder.appName("ModeloPreditivo").getOrCreate()

In [3]:
# Criar um DataFrame de exemplo
dados = spark.createDataFrame([
    (1, 35, 60000, 1),
    (0, 45, 50000, 0),
    (1, 28, 75000, 1),
    (0, 50, 80000, 0),
    (1, 32, 65000, 1),
    (0, 55, 70000, 0)
], ["genero", "idade", "renda", "comprou"])

In [4]:
# Preparar os dados
colunas_caracteristicas = ["genero", "idade", "renda"]
montador = VectorAssembler(inputCols=colunas_caracteristicas, outputCol="caracteristicas")
dados = montador.transform(dados)

In [5]:
# Dividir os dados em conjuntos de treinamento e teste
(dados_treino, dados_teste) = dados.randomSplit([0.7, 0.3], seed=12345)

# Criar e treinar o modelo de regressão logística
rl = LogisticRegression(featuresCol="caracteristicas", labelCol="comprou")
modelo = rl.fit(dados_treino)

# Fazer previsões no conjunto de teste
previsoes = modelo.transform(dados_teste)

# Avaliar o modelo
avaliador = BinaryClassificationEvaluator(labelCol="comprou", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
auc = avaliador.evaluate(previsoes)

print(f"Área sob a curva ROC: {auc}")

Área sob a curva ROC: 1.0


In [6]:
# Mostrar algumas previsões
previsoes.select("genero", "idade", "renda", "comprou", "prediction", "probability").show()

# Parar a sessão Spark
spark.stop()

+------+-----+-----+-------+----------+--------------------+
|genero|idade|renda|comprou|prediction|         probability|
+------+-----+-----+-------+----------+--------------------+
|     0|   55|70000|      0|       0.0|[0.99999999998361...|
|     1|   32|65000|      1|       1.0|[7.04727709059451...|
+------+-----+-----+-------+----------+--------------------+

